<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;margin-bottom:20px">
  <h1 style="color:#ffffff;font-size:2.2em;margin:0 0 8px 0">
    🎯 Optimización de Modelos QSAR con Optuna
  </h1>
  <p style="color:#a8c4e0;font-size:1.1em;margin:0">
    NB-ML-02 · Búsqueda de hiperparámetros · EGFR (CHEMBL203) · UNAL 2026
  </p>
</div>


---
## ¿Qué es Optuna y por qué usarlo?

**Optuna** es un framework de optimización de hiperparámetros basado en
**búsqueda bayesiana**: en lugar de probar combinaciones al azar (RandomSearch)
o todas las combinaciones posibles (GridSearch), aprende de los intentos anteriores
y dirige la búsqueda hacia las regiones del espacio que han dado mejores resultados.

| Método | Estrategia | Intentos necesarios | Velocidad |
|--------|-----------|---------------------|-----------|
| GridSearch | Todas las combinaciones | Alto (exponencial) | Lento |
| RandomSearch | Aleatorio | Medio | Rápido |
| **Optuna (TPE)** | Bayesiana | **Bajo** | **Rápido** |

En este notebook:
1. Cargamos el split y los resultados del **NB-ML-01** (08_-_Modelos_QSAR)
2. Optimizamos RF y XGBoost con Optuna
3. Comparamos el modelo optimizado vs el baseline del NB-ML-01
4. Guardamos el mejor modelo para producción


---
## 1. Instalación y carga de datos

In [ ]:
# ── Instalar librerías ───────────────────────────────────────────────────────
!pip install optuna xgboost scikit-learn imbalanced-learn joblib --quiet
!pip install optuna-integration --quiet   # integración sklearn

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silenciar logs verbosos

print(f"✅ Optuna {optuna.__version__} listo")


In [ ]:
# ── Importaciones ────────────────────────────────────────────────────────────
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os, json, pickle, joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.metrics         import (roc_auc_score, f1_score,
                                     matthews_corrcoef, accuracy_score,
                                     confusion_matrix, ConfusionMatrixDisplay,
                                     roc_curve, classification_report)
try:
    import xgboost as xgb
    XGB_OK = True
    print(f"✅ XGBoost {xgb.__version__}")
except ImportError:
    XGB_OK = False
    print("⚠️  XGBoost no disponible")

print("✅ Importaciones completadas")


---
## 2. Cargar datos desde GitHub

Cargamos los mismos archivos que en el NB-ML-01 para garantizar que la optimización usa exactamente el mismo split.

In [ ]:
# ── URLs del repositorio del curso ──────────────────────────────────────────
import requests, io

BASE = "https://raw.githubusercontent.com/FelPVic/curso_datascience/main/files"
URL_LABELS = f"{BASE}/labels_egfr_(chembl203).csv"
URL_DESC   = f"{BASE}/features_descriptores_egfr_(chembl203).csv"

TARGET_NAME        = "EGFR (CHEMBL203)"
UMBRAL_PACTIVIDAD  = 8.5
TARGET_SLUG        = "egfr_(chembl203)"

# Labels
ARCHIVO_LABELS = next((f for f in os.listdir('.') if f.startswith('labels_') and f.endswith('.csv')), None)
if ARCHIVO_LABELS:
    df_labels = pd.read_csv(ARCHIVO_LABELS)
    print(f"✅ Labels cargados localmente: {ARCHIVO_LABELS}")
else:
    print("Descargando labels...")
    df_labels = pd.read_csv(URL_LABELS)

# Descriptores
ARCHIVO_DESC = next((f for f in os.listdir('.') if f.startswith('features_descriptores_') and f.endswith('.csv')), None)
if ARCHIVO_DESC:
    df_desc_raw = pd.read_csv(ARCHIVO_DESC)
    print(f"✅ Descriptores cargados localmente: {ARCHIVO_DESC}")
else:
    print("Descargando descriptores (~83 MB, puede tardar 2-4 min)...")
    df_desc_raw = pd.read_csv(URL_DESC)


In [ ]:
import requests, io, pickle, json

BASE = "https://raw.githubusercontent.com/FelPVic/curso_datascience/main/files/modelos"

# ── Primero ver qué hay en el meta.json ──────────────────────────────────────
r_meta = requests.get(f"{BASE}/meta_egfr_chembl203.json", timeout=30)
meta   = json.loads(r_meta.text)
print("Contenido del meta.json:")
print(json.dumps(meta, indent=2, ensure_ascii=False))

In [ ]:
# ── Cargar modelos individuales ───────────────────────────────────────────────
modelos_urls = {
    'Reg. Logística': f"{BASE}/lr_egfr_chembl203.pkl",
    'SVM (RBF)':      f"{BASE}/svm_egfr_chembl203.pkl",
    'Random Forest':  f"{BASE}/rf_egfr_chembl203.pkl",
    'XGBoost':        f"{BASE}/xgb_egfr_chembl203.pkl",
}

modelos = {}
print("Cargando modelos desde GitHub...")
print("=" * 50)
for nombre, url in modelos_urls.items():
    try:
        r       = requests.get(url, timeout=60)
        modelo  = pickle.load(io.BytesIO(r.content))
        modelos[nombre] = modelo
        tam     = len(r.content) / 1024 / 1024
        tipo    = type(modelo).__name__
        print(f"  ✅ {nombre:<20} ({tam:.1f} MB)  tipo: {tipo}")
    except Exception as e:
        print(f"  ❌ {nombre:<20} Error: {e}")

# El meta.json tiene los nombres de features y métricas
NOMBRES_FEATURES = meta.get('nombres_features', [])
TARGET_NAME      = meta.get('target', 'EGFR (CHEMBL203)')

print(f"\n✅ {len(modelos)} modelos cargados")
print(f"   Features: {len(NOMBRES_FEATURES)}")

# Acceder a un modelo específico para optimizar
pipeline_final = modelos['Random Forest']   # o el que quieras optimizar
print(f"\nModelo para optimizar: {type(pipeline_final).__name__}")
print(f"Parámetros actuales:   {pipeline_final.get_params()}")

In [ ]:
# ── Reconstruir X e y con el mismo umbral del NB-ML-01 ──────────────────────
if 'pActividad_mediana' in df_labels.columns and 'pActividad' not in df_labels.columns:
    df_labels = df_labels.rename(columns={'pActividad_mediana': 'pActividad'})
df_labels = df_labels.dropna(subset=['pActividad']).reset_index(drop=True)
df_labels['activo_v2'] = (df_labels['pActividad'] >= UMBRAL_PACTIVIDAD).astype(int)
y = df_labels['activo_v2'].values

COLS_META = ['molecule_chembl_id','std_smiles','canonical_smiles',
             'pca_1','pca_2','tsne_1','tsne_2','umap_1','umap_2',
             'activo','activo_v2','pActividad','IC50_nM','standard_type',
             'scaffold','cluster','MW','MolLogP','QED']
FEATURE_COLS = [c for c in df_desc_raw.columns
                if c not in COLS_META
                and pd.api.types.is_numeric_dtype(df_desc_raw[c])]

X_desc = (df_desc_raw[FEATURE_COLS]
          .apply(pd.to_numeric, errors='coerce')
          .fillna(df_desc_raw[FEATURE_COLS].median())
          .values)
NOMBRES_FEATURES = FEATURE_COLS

# Split idéntico al NB-ML-01 (misma semilla)
X_train, X_test, y_train, y_test = train_test_split(
    X_desc, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Split reconstruido:")
print(f"   Train: {len(y_train)}  ({y_train.sum()} activos, {(y_train==0).sum()} inactivos)")
print(f"   Test:  {len(y_test)}   ({y_test.sum()} activos, {(y_test==0).sum()} inactivos)")
print(f"   Features: {X_train.shape[1]}")


---
## 3. Métricas baseline del NB-ML-01

Antes de optimizar, establecemos los valores de referencia del NB-ML-01 para poder medir la mejora real.

In [ ]:
# ── Cargar reporte del NB-ML-01 desde GitHub (si existe) ────────────────────
URL_REPORTE = f"{BASE}/reporte_modelos_{TARGET_SLUG}.csv"

try:
    df_baseline = pd.read_csv(URL_REPORTE, index_col=0)
    print("✅ Reporte NB-ML-01 cargado desde GitHub:")
    print(df_baseline.round(4).to_string())
except Exception:
    print("⚠️  Reporte no encontrado en GitHub.")
    print("   Entrenaremos los modelos base aquí para tener referencia.")
    # Entrenar baseline rápido
    scale_pos = (y_train == 0).sum() / y_train.sum()
    rf_base = RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                     class_weight='balanced', n_jobs=-1,
                                     random_state=42)
    rf_base.fit(X_train, y_train)
    auc_base = roc_auc_score(y_test, rf_base.predict_proba(X_test)[:,1])
    df_baseline = pd.DataFrame({'AUC-ROC': [auc_base]},
                               index=['Random Forest (base)'])
    print(f"  Random Forest base AUC: {auc_base:.4f}")

AUC_BASELINE = df_baseline['AUC-ROC'].max()
MEJOR_BASE   = df_baseline['AUC-ROC'].idxmax()
print(f"\n🎯 Objetivo: superar AUC = {AUC_BASELINE:.4f} ({MEJOR_BASE})")


---
## 4. Función de evaluación común

In [ ]:
# ── Función de evaluación ────────────────────────────────────────────────────
CV_INNER = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluar(nombre, modelo, X_tr, y_tr, X_te, y_te):
    """Evalúa un modelo y devuelve dict con métricas."""
    modelo.fit(X_tr, y_tr)
    y_pred  = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:,1]

    auc = roc_auc_score(y_te, y_proba)
    f1  = f1_score(y_te, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_te, y_pred)
    acc = accuracy_score(y_te, y_pred)

    print(f"  {nombre:<35} AUC={auc:.4f}  F1={f1:.4f}  MCC={mcc:.4f}")
    return {'Modelo': nombre, 'AUC-ROC': auc, 'F1': f1, 'MCC': mcc, 'Accuracy': acc}

resultados_opt = []
print("✅ Función de evaluación lista")


---
## 5. Optimización de Random Forest con Optuna

Optuna prueba distintas combinaciones de hiperparámetros usando el algoritmo **TPE**
(Tree-structured Parzen Estimator): un modelo probabilístico que aprende qué regiones
del espacio de hiperparámetros son prometedoras y concentra los intentos ahí.


In [ ]:
# ── Espacio de búsqueda — Random Forest ─────────────────────────────────────
def objetivo_rf(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators',   100, 800, step=50),
        'max_depth':       trial.suggest_int('max_depth',        3,  30),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf', 1,  20),
        'min_samples_split':trial.suggest_int('min_samples_split',2, 20),
        'max_features':    trial.suggest_categorical('max_features',
                               ['sqrt', 'log2', 0.3, 0.5]),
        'class_weight':    'balanced',
        'n_jobs':          -1,
        'random_state':    42,
    }
    modelo = RandomForestClassifier(**params)
    # AUC media en CV sobre el training set (no toca el test)
    scores = cross_val_score(modelo, X_train, y_train,
                             cv=CV_INNER, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

# ── Ejecutar la optimización ─────────────────────────────────────────────────
N_TRIALS_RF = 20   # aumentar para mejor resultado (100-200 en producción)

study_rf = optuna.create_study(
    direction='maximize',
    study_name='rf_qsar_egfr',
    sampler=optuna.samplers.TPESampler(seed=42)
)

print(f"Optimizando Random Forest ({N_TRIALS_RF} intentos)...")
print("Esto puede tardar 5-15 minutos según el hardware...")
print()
study_rf.optimize(objetivo_rf, n_trials=N_TRIALS_RF,
                  show_progress_bar=True)

print(f"\n✅ Optimización completada")
print(f"   Mejor AUC CV: {study_rf.best_value:.4f}")
print(f"   Mejores hiperparámetros:")
for k, v in study_rf.best_params.items():
    print(f"     {k:<25}: {v}")


In [ ]:
# ── Cargar reporte de optimización desde GitHub ───────────────────────────────
import requests, json
import pandas as pd
import matplotlib.pyplot as plt

URL_OPT = ("https://raw.githubusercontent.com/FelPVic/curso_datascience/"
           f"main/files/optimizacion_{TARGET_SLUG}.json")

r       = requests.get(URL_OPT, timeout=30)
reporte = json.loads(r.text)

# Reconstruir DataFrames
df_baseline = pd.DataFrame(reporte['baseline']).T
df_opt      = pd.DataFrame(reporte['optimizado']).T
AUC_BASELINE = df_baseline['AUC-ROC'].max()

# Reconstruir historia para el gráfico
historia_rf  = reporte['historia_optuna']['Random Forest']['valores']
historia_xgb = reporte['historia_optuna']['XGBoost']['valores'] if reporte['historia_optuna']['XGBoost'] else []

print(f"✅ Reporte cargado: {reporte['target']}")
print(f"   Train: {reporte['n_train']}  |  Test: {reporte['n_test']}")
print()
print("BASELINE:")
print(df_baseline.round(4).to_string())
print()
print("OPTIMIZADO:")
print(df_opt.round(4).to_string())
print()
print("MEJORES PARÁMETROS RF:")
for k, v in reporte['mejores_params']['Random Forest'].items():
    print(f"  {k:<25}: {v}")

In [ ]:
# ── Visualizar la historia de optimización ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Historia de intentos
valores = [t.value for t in study_rf.trials]
axes[0].plot(valores, alpha=0.4, color='steelblue', linewidth=0.8, label='AUC por intento')
running_max = pd.Series(valores).cummax()
axes[0].plot(running_max, color='#27ae60', linewidth=2, label='Mejor acumulado')
axes[0].axhline(AUC_BASELINE, color='#e74c3c', linestyle='--',
                linewidth=1.5, label=f'Baseline NB-ML-01 ({AUC_BASELINE:.3f})')
axes[0].set_xlabel('Intento', fontsize=11)
axes[0].set_ylabel('AUC-ROC (CV)', fontsize=11)
axes[0].set_title('Historia de optimización — Random Forest', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Importancia de hiperparámetros
try:
    importancias_hp = optuna.importance.get_param_importances(study_rf)
    params_names = list(importancias_hp.keys())
    params_vals  = list(importancias_hp.values())
    axes[1].barh(params_names[::-1], params_vals[::-1],
                 color='steelblue', alpha=0.8, edgecolor='white')
    axes[1].set_xlabel('Importancia del hiperparámetro', fontsize=11)
    axes[1].set_title('¿Qué hiperparámetro importa más?', fontsize=11)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
except Exception:
    axes[1].text(0.5, 0.5, 'Importancia no disponible\n(ejecutar más trials)',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('optuna_rf_historia.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Entrenar RF optimizado sobre todo el training set ────────────────────────
rf_opt = RandomForestClassifier(
    **study_rf.best_params,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

res_rf_opt = evaluar('Random Forest (Optuna)', rf_opt,
                      X_train, y_train, X_test, y_test)
resultados_opt.append(res_rf_opt)

mejora_auc = res_rf_opt['AUC-ROC'] - AUC_BASELINE
print(f"\n  Mejora vs baseline: {mejora_auc:+.4f} AUC")
print(f"  {'✅ Mejoró' if mejora_auc > 0 else '⚠️  No superó el baseline'}")


---
## 6. Optimización de XGBoost con Optuna

XGBoost tiene más hiperparámetros que RF y es más sensible a su configuración.
Optuna es especialmente útil aquí porque el espacio de búsqueda es continuo
y las interacciones entre parámetros son complejas.


In [ ]:
# ── Espacio de búsqueda — XGBoost ───────────────────────────────────────────
if XGB_OK:
    scale_pos = float((y_train == 0).sum() / y_train.sum())

    def objetivo_xgb(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators',    100, 600, step=50),
            'max_depth':         trial.suggest_int('max_depth',          3,  10),
            'learning_rate':     trial.suggest_float('learning_rate',  0.01, 0.3, log=True),
            'subsample':         trial.suggest_float('subsample',       0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',0.4, 1.0),
            'min_child_weight':  trial.suggest_int('min_child_weight',   1,  10),
            'gamma':             trial.suggest_float('gamma',           0.0, 5.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',       0.0, 2.0),
            'reg_lambda':        trial.suggest_float('reg_lambda',      0.5, 5.0),
            'scale_pos_weight':  scale_pos,
            'use_label_encoder': False,
            'eval_metric':       'logloss',
            'random_state':      42,
            'n_jobs':            -1,
        }
        modelo = xgb.XGBClassifier(**params)
        scores = cross_val_score(modelo, X_train, y_train,
                                 cv=CV_INNER, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    N_TRIALS_XGB = 20

    study_xgb = optuna.create_study(
        direction='maximize',
        study_name='xgb_qsar_egfr',
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    print(f"Optimizando XGBoost ({N_TRIALS_XGB} intentos)...")
    study_xgb.optimize(objetivo_xgb, n_trials=N_TRIALS_XGB,
                       show_progress_bar=True)

    print(f"\n✅ Optimización completada")
    print(f"   Mejor AUC CV: {study_xgb.best_value:.4f}")
    print(f"   Mejores hiperparámetros:")
    for k, v in study_xgb.best_params.items():
        print(f"     {k:<25}: {v}")
else:
    print("⚠️  XGBoost no disponible — omitir esta sección")


In [ ]:
# ── Visualizar historia XGBoost ──────────────────────────────────────────────
if XGB_OK:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    valores_xgb = [t.value for t in study_xgb.trials]
    axes[0].plot(valores_xgb, alpha=0.4, color='#f78166', linewidth=0.8)
    axes[0].plot(pd.Series(valores_xgb).cummax(), color='#e74c3c',
                 linewidth=2, label='Mejor acumulado')
    axes[0].axhline(AUC_BASELINE, color='#94a3b8', linestyle='--',
                    linewidth=1.5, label=f'Baseline ({AUC_BASELINE:.3f})')
    axes[0].set_xlabel('Intento', fontsize=11)
    axes[0].set_ylabel('AUC-ROC (CV)', fontsize=11)
    axes[0].set_title('Historia de optimización — XGBoost', fontsize=11)
    axes[0].legend(fontsize=9)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    try:
        imp_xgb = optuna.importance.get_param_importances(study_xgb)
        axes[1].barh(list(imp_xgb.keys())[::-1],
                     list(imp_xgb.values())[::-1],
                     color='#f78166', alpha=0.8, edgecolor='white')
        axes[1].set_xlabel('Importancia del hiperparámetro', fontsize=11)
        axes[1].set_title('¿Qué hiperparámetro importa más?', fontsize=11)
        axes[1].spines['top'].set_visible(False)
        axes[1].spines['right'].set_visible(False)
    except Exception:
        pass

    plt.tight_layout()
    plt.savefig('optuna_xgb_historia.png', dpi=130, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Entrenar XGBoost optimizado ──────────────────────────────────────────────
if XGB_OK:
    xgb_opt = xgb.XGBClassifier(
        **study_xgb.best_params,
        scale_pos_weight=scale_pos,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

    res_xgb_opt = evaluar('XGBoost (Optuna)', xgb_opt,
                           X_train, y_train, X_test, y_test)
    resultados_opt.append(res_xgb_opt)

    mejora_xgb = res_xgb_opt['AUC-ROC'] - AUC_BASELINE
    print(f"\n  Mejora vs baseline: {mejora_xgb:+.4f} AUC")


In [ ]:
# ── Calcular métricas completas del baseline desde los modelos guardados ──────
from sklearn.metrics import (roc_auc_score, f1_score, matthews_corrcoef,
                              accuracy_score)

print("Calculando métricas completas del baseline...")
print("=" * 55)

resultados_baseline = []

for nombre, modelo in modelos.items():
    try:
        y_pred  = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y_test, y_proba)
        f1  = f1_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)

        resultados_baseline.append({
            'Modelo':    nombre,
            'AUC-ROC':   auc,
            'F1':        f1,
            'MCC':       mcc,
            'Accuracy':  acc,
        })
        print(f"  {nombre:<20} AUC={auc:.4f}  F1={f1:.4f}  MCC={mcc:.4f}")

    except Exception as e:
        print(f"  ⚠️  {nombre}: {e}")

df_baseline = pd.DataFrame(resultados_baseline).set_index('Modelo')
AUC_BASELINE = df_baseline['AUC-ROC'].max()
MEJOR_BASE   = df_baseline['AUC-ROC'].idxmax()

print()
print(df_baseline.round(4).to_string())
print(f"\n🎯 Mejor baseline: {MEJOR_BASE}  AUC = {AUC_BASELINE:.4f}")

In [ ]:
# ── Guardar métricas de optimización para GitHub ─────────────────────────────
import json, os
import numpy as np

def convertir_serializable(obj):
    """Convierte tipos numpy a tipos Python nativos para JSON."""
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

# ── Construir el reporte completo ────────────────────────────────────────────
reporte = {
    'target':          TARGET_NAME,
    'umbral_pact':     float(UMBRAL_PACTIVIDAD),
    'n_train':         int(len(y_train)),
    'n_test':          int(len(y_test)),

    'baseline': {
        nombre: {
            'AUC-ROC':  float(row['AUC-ROC']),
            'F1':       float(row['F1']),
            'MCC':      float(row['MCC']),
            'Accuracy': float(row['Accuracy']),
        }
        for nombre, row in df_baseline.iterrows()
    },

    'optimizado': {
        nombre: {
            'AUC-ROC':  float(row['AUC-ROC']),
            'F1':       float(row['F1']),
            'MCC':      float(row['MCC']),
            'Accuracy': float(row['Accuracy']),
        }
        for nombre, row in df_opt.iterrows()
    },

    'mejores_params': {
        'Random Forest': {
            k: convertir_serializable(v)
            for k, v in study_rf.best_params.items()
        },
        'XGBoost': {
            k: convertir_serializable(v)
            for k, v in study_xgb.best_params.items()
        } if XGB_OK else {},
    },

    'historia_optuna': {
        'Random Forest': {
            'n_trials':   len(study_rf.trials),
            'mejor_auc_cv': float(study_rf.best_value),
            'valores':    [float(t.value) for t in study_rf.trials],
        },
        'XGBoost': {
            'n_trials':   len(study_xgb.trials),
            'mejor_auc_cv': float(study_xgb.best_value),
            'valores':    [float(t.value) for t in study_xgb.trials],
        } if XGB_OK else {},
    },
}

# ── Guardar ───────────────────────────────────────────────────────────────────
archivo = f'optimizacion_{TARGET_SLUG}.json'
with open(archivo, 'w', encoding='utf-8') as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)

tam = os.path.getsize(archivo) / 1024
print(f"✅ Guardado: {archivo}  ({tam:.1f} KB)")
print()

# Preview
print("Contenido del reporte:")
print(f"  Baseline:    {list(reporte['baseline'].keys())}")
print(f"  Optimizado:  {list(reporte['optimizado'].keys())}")
print(f"  Parámetros:  RF ({len(reporte['mejores_params']['Random Forest'])} params)")
if XGB_OK:
    print(f"               XGB ({len(reporte['mejores_params']['XGBoost'])} params)")
print(f"  Historia RF: {reporte['historia_optuna']['Random Forest']['n_trials']} trials")

---
## 7. Comparación: baseline vs modelos optimizados

In [ ]:
# ── Tabla comparativa final ───────────────────────────────────────────────────
df_opt = pd.DataFrame(resultados_opt).set_index('Modelo')

# Renombrar baseline para distinguirlo
df_baseline_comp = df_baseline.copy()
df_baseline_comp.index = [f"{i} (baseline)" for i in df_baseline_comp.index]

df_comp = pd.concat([df_baseline_comp, df_opt])

print("COMPARACIÓN: BASELINE vs MODELOS OPTIMIZADOS")
print("=" * 65)
print(df_comp.round(4).to_string())

mejor_opt = df_opt['AUC-ROC'].idxmax()
print(f"\n✅ Mejor modelo optimizado: {mejor_opt}")
print(f"   AUC = {df_opt.loc[mejor_opt, 'AUC-ROC']:.4f}")
print(f"   Mejora vs Random Forest baseline: "
      f"{df_opt.loc[mejor_opt,'AUC-ROC'] - AUC_BASELINE:+.4f}")


In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 6))
metricas = ['AUC-ROC', 'F1', 'MCC']

# Color por origen: baseline=azul, optimizado=verde
colores = ['#3d6b99' if 'baseline' in m else '#27ae60'
           for m in df_comp.index]

for ax, metrica in zip(axes, metricas):
    vals = df_comp[metrica].sort_values()
    bar_colors = [colores[list(df_comp.index).index(i)] for i in vals.index]
    bars = ax.barh(vals.index.tolist(), vals,
                   color=bar_colors, alpha=0.85, edgecolor='white')
    ax.set_xlabel(metrica, fontsize=11)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    for bar, val in zip(bars, vals):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3d6b99', label='Baseline NB-ML-01'),
                   Patch(facecolor='#27ae60', label='Optimizado (Optuna)')]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))
plt.suptitle(f'Baseline vs Optimizado — {TARGET_NAME}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_optuna.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Curvas ROC comparativas ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

# Colores por modelo
palette = {
    'Reg. Logística': '#94a3b8',
    'SVM (RBF)':      '#7c3aed',
    'Random Forest':  '#0ea5e9',
    'XGBoost':        '#f97316',
}

# Curvas baseline (línea punteada)
for nombre, modelo in modelos.items():
    fpr, tpr, _ = roc_curve(y_test, modelo.predict_proba(X_test)[:,1])
    auc = df_baseline.loc[nombre, 'AUC-ROC']
    ax.plot(fpr, tpr, color=palette.get(nombre, 'gray'),
            linewidth=1.5, linestyle='--', alpha=0.6,
            label=f'{nombre} base (AUC={auc:.3f})')

# Curvas optimizadas (línea sólida)
modelos_opt_map = {}
if 'rf_opt' in dir():
    rf_opt.fit(X_train, y_train)
    modelos_opt_map['Random Forest (Optuna)'] = (rf_opt, '#0ea5e9')
if XGB_OK and 'xgb_opt' in dir():
    xgb_opt.fit(X_train, y_train)
    modelos_opt_map['XGBoost (Optuna)'] = (xgb_opt, '#f97316')

for nombre_opt, (modelo_opt, color) in modelos_opt_map.items():
    fpr, tpr, _ = roc_curve(y_test, modelo_opt.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, modelo_opt.predict_proba(X_test)[:,1])
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{nombre_opt} (AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.4, label='Azar')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11)
ax.set_title(f'Curvas ROC — Baseline vs Optimizado\n{TARGET_NAME}', fontsize=11)
ax.legend(fontsize=8, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('roc_comparacion_completa.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 8. Guardar el mejor modelo optimizado

In [ ]:
# ── Seleccionar el mejor modelo optimizado ───────────────────────────────────
if XGB_OK and res_xgb_opt['AUC-ROC'] >= res_rf_opt['AUC-ROC']:
    mejor_modelo_opt = xgb_opt
    mejor_nombre_opt = 'XGBoost (Optuna)'
    mejor_params_opt = study_xgb.best_params
else:
    mejor_modelo_opt = rf_opt
    mejor_nombre_opt = 'Random Forest (Optuna)'
    mejor_params_opt = study_rf.best_params

# Re-entrenar con TODO el dataset (train + test)
mejor_modelo_opt.fit(X_desc, y)
print(f"✅ Mejor modelo re-entrenado con todos los datos: {mejor_nombre_opt}")


In [ ]:
# ── Guardar archivos ─────────────────────────────────────────────────────────
import os, json, joblib

# 1. Modelo optimizado
archivo_modelo_opt = f'modelo_optimizado_{TARGET_SLUG}.pkl'
joblib.dump({
    'modelo':           mejor_modelo_opt,
    'nombres_features': NOMBRES_FEATURES,
    'target':           TARGET_NAME,
    'nombre_modelo':    mejor_nombre_opt,
    'hiperparametros':  mejor_params_opt,
    'metricas':         df_opt.to_dict(),
    'umbral_pact':      UMBRAL_PACTIVIDAD,
    'umbral_decision':  0.5,
}, archivo_modelo_opt)

# 2. Hiperparámetros en JSON (legible)
archivo_params = f'best_params_{TARGET_SLUG}.json'
params_guardar = {
    'target':        TARGET_NAME,
    'modelo':        mejor_nombre_opt,
    'auc_cv_optuna': study_rf.best_value if 'RF' in mejor_nombre_opt
                     else study_xgb.best_value,
    'auc_test':      df_opt.loc[mejor_nombre_opt, 'AUC-ROC'],
    'auc_baseline':  AUC_BASELINE,
    'mejora':        df_opt.loc[mejor_nombre_opt, 'AUC-ROC'] - AUC_BASELINE,
    'hiperparametros': {k: (float(v) if isinstance(v, (np.floating, np.integer))
                             else v)
                        for k, v in mejor_params_opt.items()},
    'n_trials_rf':   N_TRIALS_RF,
    'n_trials_xgb':  N_TRIALS_XGB if XGB_OK else 0,
}
with open(archivo_params, 'w') as f:
    json.dump(params_guardar, f, indent=2, ensure_ascii=False)

# 3. Reporte comparativo
archivo_comp = f'reporte_optimizacion_{TARGET_SLUG}.csv'
df_comp.round(4).to_csv(archivo_comp)

print("✅ ARCHIVOS GUARDADOS")
print("=" * 55)
for archivo in [archivo_modelo_opt, archivo_params, archivo_comp]:
    tam = os.path.getsize(archivo) / 1024 / 1024
    print(f"  {archivo:<45} ({tam:.1f} MB)")

print()
print("Para subir a GitHub:")
print("  → Los .json y .csv van directamente (sin Git LFS)")
print("  → El .pkl grande puede necesitar Git LFS o Google Drive")


---
## ✅ Resumen del notebook

| Sección | Técnica | Resultado |
|---------|---------|-----------|
| **2. Carga** | Mismos datos y split que NB-ML-01 | Reproducibilidad garantizada |
| **3. Baseline** | Métricas del NB-ML-01 | Referencia para medir mejora |
| **5. RF Optuna** | TPE, 50 trials, CV 5-fold | `study_rf.best_params` |
| **6. XGB Optuna** | TPE, 50 trials, CV 5-fold | `study_xgb.best_params` |
| **7. Comparación** | Tabla + curvas ROC | Mejora medida vs baseline |
| **8. Guardado** | joblib + JSON + CSV | Listo para GitHub / Drive |

### Archivos generados

| Archivo | Uso |
|---------|-----|
| `modelo_optimizado_egfr_(chembl203).pkl` | Modelo listo para predicción |
| `best_params_egfr_(chembl203).json` | Hiperparámetros (legible, sin Git LFS) |
| `reporte_optimizacion_egfr_(chembl203).csv` | Tabla comparativa |

### Para continuar

```python
import joblib
datos = joblib.load('modelo_optimizado_egfr_(chembl203).pkl')
modelo      = datos['modelo']
features    = datos['nombres_features']
# → usar con la función predecir_actividad() del NB-ML-01
```
